In [ ]:
import sys
sys.path.append('../')

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from models.model import STGAT
from train import prepare_data, NYCTaxiDataset
from torch.utils.data import DataLoader
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import pandas as pd


# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# 1. Load arrays 
node_features, demand_data, train_indices, test_indices, num_nodes, num_features = prepare_data('../processed/final_stgat_input.csv', seq_len=48)

# Load graph edges
adj_matrix = np.load('../processed/adjacency_matrix.npy')
edges = np.argwhere(adj_matrix == 1)
edge_index = torch.tensor(edges.T, dtype=torch.long)

# Initialize the model and load the trained weights
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
edge_index = torch.tensor(edges.T, dtype=torch.long).to(device)

# Create Test DataLoader 
test_dataset = NYCTaxiDataset(node_features, demand_data, test_indices, seq_len=48)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False) # NEVER shuffle test data!

# Initialize model and load winning weights from grid search
model = STGAT(in_channels=num_features, hidden_channels=64, out_channels=1, num_nodes=num_nodes).to(device)
model.load_state_dict(torch.load('../models/stgat_weights_best.pth', weights_only=True))
model.eval() 

print("\nModel and Data successfully loaded!")

In [ ]:
print("Running ST-GAT model on June 2024 Test Data...")
 
stgat_predictions = []
baseline_predictions = []
actual_truths = []

with torch.no_grad():
    for batch_X, batch_Y in test_loader:
        # Move to GPU
        batch_X = batch_X.to(device)
        
        # ST-GAT forward pass
        with torch.amp.autocast('cuda'):
            batch_pred = model(batch_X, edge_index)
            
        stgat_predictions.append(batch_pred.cpu().numpy())
        
        # We also collect the actual truths from the dataloader here
        actual_truths.append(batch_Y.numpy())
        
        # Naive Baseline calculation
        base_pred = batch_X[:, -1, :, 0:1].cpu().numpy()
        baseline_predictions.append(base_pred)

# Concatenate all batch predictions into a single array
stgat_predictions = np.concatenate(stgat_predictions, axis=0)
baseline_predictions = np.concatenate(baseline_predictions, axis=0)

# Create Y_test_numpy from the collected truths
Y_test_numpy = np.concatenate(actual_truths, axis=0)

# Calculate MSE / MAE
def calculate_metrics(pred, truth):
    mse = np.mean((pred - truth)**2)
    mae = np.mean(np.abs(pred - truth))
    return mse, mae

stgat_mse, stgat_mae = calculate_metrics(stgat_predictions, Y_test_numpy)
base_mse, base_mae = calculate_metrics(baseline_predictions, Y_test_numpy)

print(f"[   MEAN SQUARED ERROR (MSE)   ]")
print(f"Naive Baseline: {base_mse:.6f}")
print(f"ST-GAT Model:   {stgat_mse:.6f}")

print(f"\n[   MEAN ABSOLUTE ERROR (MAE)   ]")
print(f"Naive Baseline: {base_mae:.6f}")
print(f"ST-GAT Model:   {stgat_mae:.6f}")

# Load scaler 
with open('../processed/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

# Reshape predictions from 3D to 2D
preds_2d = stgat_predictions.reshape(stgat_predictions.shape[0], -1)
actuals_2d = Y_test_numpy.reshape(Y_test_numpy.shape[0], -1)

num_samples = preds_2d.shape[0]
num_total_features = 272 
num_zones = 263 
 
dummy_preds = np.zeros((num_samples, num_total_features))
dummy_actuals = np.zeros((num_samples, num_total_features))
 
dummy_preds[:, :num_zones] = preds_2d
dummy_actuals[:, :num_zones] = actuals_2d
 
preds_real_2d = scaler.inverse_transform(dummy_preds)[:, :num_zones]
actuals_real_2d = scaler.inverse_transform(dummy_actuals)[:, :num_zones]

# Flatten back out for WMAPE math
preds_real = preds_real_2d.flatten()
actuals_real = actuals_real_2d.flatten()

# Ensure we don't have negative taxi rides from minor model undershoots
preds_real = np.maximum(preds_real, 0)
actuals_real = np.maximum(actuals_real, 0)

# Calculate WMAPE on real numbers
sum_absolute_errors = np.sum(np.abs(actuals_real - preds_real))
sum_actuals = np.sum(actuals_real)

# Protect against dividing by zero
if sum_actuals == 0:
    wmape = 0
else:
    wmape = (sum_absolute_errors / sum_actuals) * 100
    
accuracy = 100 - wmape

print("\n[   PERCENTAGE ACCURACY   ]")
print(f"Error (WMAPE): {wmape:.2f}%")
print(f"Prediction Accuracy: {accuracy:.2f}%")

In [ ]:
# Calculate RMSE on real-world taxi counts
rmse = np.sqrt(np.mean((actuals_real - preds_real)**2))
mae_real = np.mean(np.abs(actuals_real - preds_real))

print("[  MAGNITUDE OF ERROR (REAL TAXI COUNTS)  ]")
print(f"Mean Absolute Error (MAE): {mae_real:.2f} rides per zone/interval")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} rides per zone/interval")

# The difference between MAE and RMSE shows how much outliers are hurting our model's performance.
# If RMSE is significantly higher than MAE, the model has occasional massive misses which are not optimal.

In [ ]:
print("[   CONTEXTUAL PERFORMANCE: RUSH HOUR VS. QUIET HOUR   ]")

# Create an array of shape that represents time step of the day 
time_of_day = np.arange(num_samples) % 96
 
# Morning Rush is 7:00 AM to 10:00 AM 
# Evening Rush is 4:00 PM to 7:00 PM 
rush_hour_mask = ((time_of_day >= 28) & (time_of_day < 40)) | ((time_of_day >= 64) & (time_of_day < 76))

# Quiet Hours are 12:00 AM to 5:00 AM 
quiet_hour_mask = (time_of_day >= 0) & (time_of_day < 20)

# Helper function to calculate WMAPE for clean code
def calculate_wmape(actuals, preds):
    sum_abs_err = np.sum(np.abs(actuals - preds))
    sum_act = np.sum(actuals)
    if sum_act == 0:
        return 0
    return (sum_abs_err / sum_act) * 100

# Apply masks to 2D arrays to extract only specific times across all zones
rush_actuals = actuals_real_2d[rush_hour_mask]
rush_preds = preds_real_2d[rush_hour_mask]

quiet_actuals = actuals_real_2d[quiet_hour_mask]
quiet_preds = preds_real_2d[quiet_hour_mask]

# Calculate WMAPE for each context
rush_wmape = calculate_wmape(rush_actuals, rush_preds)
quiet_wmape = calculate_wmape(quiet_actuals, quiet_preds)

print(f"[Rush Hour] WMAPE: {rush_wmape:.2f}% | Accuracy: {100 - rush_wmape:.2f}%")
print(f"[Quiet Hour] WMAPE: {quiet_wmape:.2f}% | Accuracy: {100 - quiet_wmape:.2f}%")

In [ ]:
# Plotting Chart 
metrics = ['MSE', 'MAE']
baseline_scores = [base_mse, base_mae]
stgat_scores = [stgat_mse, stgat_mae]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
rects1 = ax.bar(x - width/2, baseline_scores, width, label='Naive Baseline', color='lightcoral')
rects2 = ax.bar(x + width/2, stgat_scores, width, label='ST-GAT (Ours)', color='steelblue')

ax.set_ylabel('Error (Lower is Better)', fontsize=12)
ax.set_title('Predictive Performance on 2025 Test Data', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=12)
ax.legend()

for rect in rects1 + rects2:
    height = rect.get_height() 
    ax.annotate(f'{height:.6f}',
                xy=(rect.get_x() + rect.get_width() / 2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('performance_comparison.png', dpi=300)
plt.show()

In [ ]:
# Set style
plt.style.use('seaborn-v0_8-whitegrid')

fig, ax = plt.subplots(figsize=(10, 8))

# Plot scatter with low alpha due to high volume of points (~750k)
# Creates a density effect where overlapping points become darker
ax.scatter(actuals_real, preds_real, alpha=0.05, color='steelblue', s=10)

# Draw perfect prediction line 
# Calculate max value to know how far to draw line
max_val = max(np.max(actuals_real), np.max(preds_real))
ax.plot([0, max_val], [0, max_val], color='red', linestyle='--', linewidth=2, label='Perfect Prediction (y=x)')

ax.set_title('Model Reliability: Actual vs. Predicted Taxi Demand', fontsize=16, fontweight='bold')
ax.set_xlabel('Actual Demand (Real Taxi Rides)', fontsize=14)
ax.set_ylabel('Predicted Demand (Model Output)', fontsize=14)
 
ax.set_xlim([0, np.percentile(actuals_real, 99.9)])
ax.set_ylim([0, np.percentile(preds_real, 99.9)])

ax.legend(fontsize=12)
plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
print("Calculating zone-specific errors...")

# Calculate WMAPE for each of the 263 individual zones 
zone_absolute_errors = np.sum(np.abs(actuals_real_2d - preds_real_2d), axis=0)
zone_actuals = np.sum(actuals_real_2d, axis=0)

# Calculate percentage error, avoid division by zero for empty zones
zone_wmape = np.zeros(num_zones)
for i in range(num_zones):
    if zone_actuals[i] == 0:
        zone_wmape[i] = 0 
    else:
        zone_wmape[i] = (zone_absolute_errors[i] / zone_actuals[i]) * 100

# Load Taxi Zone Shapefile 
try:
    gdf = gpd.read_file('../data/taxi_zones/taxi_zones.shp')
    
    # Create a DataFrame of errors and merge it with map 
    error_df = pd.DataFrame({
        'LocationID': range(1, num_zones + 1),
        'WMAPE': zone_wmape
    })
    
    # Merge errors into geographic data
    gdf_mapped = gdf.merge(error_df, on='LocationID', how='left')

    # Plot the Heatmap
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # Using Orange-Red colormap: Darker red = higher error
    gdf_mapped.plot(column='WMAPE', cmap='OrRd', legend=True, 
                    legend_kwds={'label': "Error (WMAPE %)", 'orientation': "vertical"},
                    edgecolor='black', linewidth=0.2, ax=ax)
    
    ax.set_title('Spatial Forecasting Error Across NYC Taxi Zones', fontsize=16, fontweight='bold')
    ax.axis('off') 
    
    plt.tight_layout()
    plt.savefig('spatial_error_heatmap.png', dpi=300, bbox_inches='tight')
    plt.show()

except FileNotFoundError:
    print("Could not find the shapefile at '../data/taxi_zones/taxi_zones.shp'. Please check the path!")

In [ ]:
# Permutation Feature Importance
print("Running Permutation Feature Importance...")
importance_scores = []

# Base MSE without any scrambling
base_mse, _ = calculate_metrics(stgat_predictions, Y_test_numpy)

for feature_idx in range(num_features):
    # Create a fresh copy of the master node_features array
    temp_node_features = node_features.copy()
    
    # Shuffle specific feature across the entire timeline
    feature_data = temp_node_features[:, :, feature_idx].flatten()
    np.random.shuffle(feature_data)
    temp_node_features[:, :, feature_idx] = feature_data.reshape(temp_node_features[:, :, feature_idx].shape)
    
    # Create temporary dataset and DataLoader with the scrambled data
    temp_dataset = NYCTaxiDataset(temp_node_features, demand_data, test_indices, seq_len=48)
    temp_loader = DataLoader(temp_dataset, batch_size=32, shuffle=False)
    
    # Run predictions using the new temp_loader
    scrambled_preds = []
    with torch.no_grad():
        for batch_X, _ in temp_loader:
            batch_X = batch_X.to(device)
            with torch.amp.autocast('cuda'):
                batch_pred = model(batch_X, edge_index)
            scrambled_preds.append(batch_pred.cpu().numpy())
            
    scrambled_preds = np.concatenate(scrambled_preds, axis=0)
     
    scrambled_mse, _ = calculate_metrics(scrambled_preds, Y_test_numpy)
    importance = scrambled_mse - base_mse
    importance_scores.append(importance)

df_temp = pd.read_csv('../processed/final_stgat_input.csv', index_col=0, nrows=0)
weather_cols = [c for c in df_temp.columns if not c.isdigit()]
feature_names = ['Past Demand'] + weather_cols

fig, ax = plt.subplots(figsize=(10, 6))

# Focus just on Exogenous/Weather features
weather_features = feature_names[1:]
weather_importances = importance_scores[1:]

ax.bar(weather_features, weather_importances, color='mediumseagreen')
ax.set_ylabel('Increase in MSE (Importance)', fontsize=12)
ax.set_title('Exogenous Feature Importance (Weather & Temporal Impacts)', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=300)
plt.show()

In [ ]:
print("[   Isolating the top 1% worst predictions   ]")

# Create exact timestamps for June 2024 
timestamps = pd.date_range(start='2024-06-01 00:00:00', periods=num_samples, freq='15min')

# Flatten arrays while aligning Time and Zone Data
# np.repeat duplicates each timestamp 
# np.tile repeats the zone sequence 
flat_timestamps = np.repeat(timestamps, num_zones)
flat_zones = np.tile(np.arange(1, num_zones + 1), num_samples)

# Build Master Error DataFrame
error_df = pd.DataFrame({
    'Datetime': flat_timestamps,
    'Zone_ID': flat_zones,
    'Actual': actuals_real,
    'Predicted': preds_real,
    'Absolute_Error': np.abs(actuals_real - preds_real)
})

# Extract Top 1% Worst Misses
top_1_percent_cutoff = int(len(error_df) * 0.01)
worst_predictions = error_df.sort_values(by='Absolute_Error', ascending=False).head(top_1_percent_cutoff)

# Analyze worst predictions
print(f"Total predictions analyzed: {len(error_df):,}")
print(f"Top 1% cutoff size: {top_1_percent_cutoff:,} predictions\n")

print("[   TOP 5 Worst Misses   ]")
print(worst_predictions.head(5).to_string(index=False))

print("\n[   Most Frequent Failures (Top 5 Hours)   ]")
worst_predictions['Hour'] = worst_predictions['Datetime'].dt.hour
print(worst_predictions['Hour'].value_counts().head(5))

print("\n[   Zones With Most Failures (Top 5 Zones)   ]")
print(worst_predictions['Zone_ID'].value_counts().head(5))